# **Hands-on Code Examples for LangChain**

### **Environment Setup**

In [1]:
# Install the core LangChain and Google Generative AI integration
!pip install -qU langchain langchain-google-genai langchain-community wikipedia

import os
from google.colab import userdata

# Securely retrieve your API key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [8]:
!pip install -qU langchain-classic langchainhub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.2 MB/s eta 0:00:00


In [12]:
import os
import warnings
import logging

# 1. Block all Python warnings globally
os.environ["PYTHONWARNINGS"] = "ignore"

# 2. Suppress warnings from the built-in library
warnings.filterwarnings("ignore")

# 3. Suppress LangChain and Pydantic logging
logging.getLogger("langchain").setLevel(logging.ERROR)
logging.getLogger("pydantic").setLevel(logging.ERROR)

print("Environment Silenced: Warnings and logs are now suppressed.")

Environment Silenced: Warnings and logs are now suppressed.


### **Basic LLM call**

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

def call_basic_llm(query: str):
    """Simple invocation using the latest Gemini 3 Flash model."""
    # Using 'gemini-2.5-flash' which is the 2026 standard for high-performance/low-cost
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

    # Passing the query as a HumanMessage
    response = llm.invoke([HumanMessage(content=query)])
    return response.content

# Execution
try:
    result = call_basic_llm("Explain LangChain orchestration.")
    print(result)
except Exception as e:
    print(f"Error: {e}")

LangChain orchestration refers to its ability to **manage and coordinate complex workflows involving Large Language Models (LLMs) and various external components** to achieve specific application goals.

Think of an orchestra conductor. The conductor doesn't play every instrument, but they ensure all instruments play together, at the right time, with the right dynamics, to produce a harmonious piece of music. LangChain acts as that conductor for your LLM applications.

**Why is Orchestration Needed for LLM Applications?**

LLMs are incredibly powerful, but they have limitations:

1.  **Lack of Real-time Data:** They are trained on a static dataset and don't inherently know current events or specific user data.
2.  **Inability to Perform Actions:** They can generate text, but they can't browse the web, execute code, query a database, or call an API on their own.
3.  **Limited Context Window:** They can only remember so much information at once.
4.  **Hallucination:** They can sometimes 

### **Prompt Template Usage**

In [5]:
from langchain_core.prompts import ChatPromptTemplate

def test_prompt_template(topic: str, tone: str):
    """Demonstrates creating and formatting a reusable ChatPromptTemplate."""
    template = ChatPromptTemplate.from_messages([
        ("system", "You are a technical mentor. Explain concepts in a {tone} tone."),
        ("human", "Explain the concept of {topic}.")
    ])

    # We can format it to see the final prompt string
    formatted_prompt = template.format_messages(topic=topic, tone=tone)
    print(f"Formatted Prompt: {formatted_prompt}")
    return template

# Execution
test_prompt_template("Prompt Engineering", "encouraging")

Formatted Prompt: [SystemMessage(content='You are a technical mentor. Explain concepts in a encouraging tone.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain the concept of Prompt Engineering.', additional_kwargs={}, response_metadata={})]


ChatPromptTemplate(input_variables=['tone', 'topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['tone'], input_types={}, partial_variables={}, template='You are a technical mentor. Explain concepts in a {tone} tone.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Explain the concept of {topic}.'), additional_kwargs={})])

### **Simple Chain (using LCEL)**

In [6]:
from langchain_core.output_parsers import StrOutputParser

def run_simple_chain(concept: str):
    """A standard pipeline: Prompt -> LLM -> String Output Parser."""
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    prompt = ChatPromptTemplate.from_template("What is the core benefit of {concept}?")

    # Building the chain using LCEL
    chain = prompt | llm | StrOutputParser()

    return chain.invoke({"concept": concept})

# Execution
print(run_simple_chain("LangChain LCEL"))

The core benefit of LangChain LCEL (LangChain Expression Language) is its ability to provide a **standardized, highly composable, and efficient way to build complex, production-ready LLM applications.**

In simpler terms, it's about making it incredibly easy to **chain together different components (LLMs, tools, retrievers, parsers, custom logic) into robust, performant, and maintainable workflows.**

Here's a breakdown of why that's the core benefit:

1.  **Standardized Interface (Runnable Protocol):** Every component in LCEL (LLMs, prompts, output parsers, retrievers, tools, custom functions) adheres to a common `Runnable` interface. This means they all have predictable `invoke`, `batch`, `stream`, and `ainvoke` methods, allowing them to be seamlessly combined.

2.  **Highly Composability (Pipe Operator `|`):** The `|` operator (similar to Unix pipes) is central. It allows you to link components together in a clear, sequential, and intuitive manner. This makes building complex chains

### **Agent with a Tool**

In [16]:
# --- AGENT WITH TOOL EXAMPLE ---
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
import langchain.agents as agents

def run_clean_agent(query: str):
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1))

    # Set verbose=False to stop the inner reasoning logs from printing
    agent = agents.create_agent(
        model=llm,
        tools=[wiki],
        system_prompt="You are a precise technical assistant."
    )

    # Run and only return the final text
    response = agent.invoke({"messages": [("user", query)]})
    return response["messages"][-1].content

# Testing
print(run_clean_agent("Summarize the LangChain v1.0 release."))

[{'type': 'text', 'text': 'I am sorry, I cannot fulfill this request. I do not have access to real-time information or specific release notes for software like LangChain v1.0. My knowledge cutoff is a while ago, and I cannot browse the internet.', 'extras': {'signature': 'CrACAb4+9vuq9xqmzQ0CwZhFVc8oHVGil1l/ZwG1/l0Rc3fC8H6TDsRLoIlUliY0/G/5kKOpfuu2q4g1xwVIFWvkpYxv5+R1Wh/Z+cYZc89lo4X+hwI87hm3KHhyqWBskztj4R6VZE8EyK88PyIvL7Iapy+FIVg8GIoPTe8EofXB/4wcg7pXaGkvgFERJggnH/9iQqDJHxXAnhRfWgUQ/q3y10UJlFMHNRd+l1hn5DxYpOLckyxypv9ydJjNIKQ9v8e4VGjEJOASBj2m0Tm4E/cn54WaEaL3vXIhWq7UGYL3Z52SNg/Dt1ReUvTRHqIK5+xhmNU/O+hRGLqxqmye4dS2C0julsVU5FWedKtZspRImsllO952VYAqsX3AEZCW69SIBpUIrhaGioEoBAN1WyUV0w=='}}]


### **Memory Example (Stateful Conversation)**

In [18]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# This dictionary stores history for different users/sessions
history_store = {}

def get_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

def chat_with_history(user_message: str, session_id: str):
    """Maintains a conversation thread using an in-memory buffer."""
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful coding assistant."),
        ("placeholder", "{history}"),
        ("human", "{input}")
    ])

    chain = prompt | llm

    wrapped_chain = RunnableWithMessageHistory(
        chain,
        get_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    return wrapped_chain.invoke(
        {"input": user_message},
        config={"configurable": {"session_id": session_id}}
    ).content

# Execution
print(chat_with_history("Hi, my name is Vedika.", "session_1"))
print(chat_with_history("Do you remember my name?", "session_1"))

Hi Vedika! It's nice to meet you. How can I help you today?
Yes, I do! Your name is Vedika.
